# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a guided template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This step initializes the dataset object and prints a summary.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the FAIR^2 dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata

print('Dataset Title:', metadata.name)
print('Description:', metadata.description)
print('Published:', getattr(metadata, 'datePublished', 'N/A'))
print('License:', getattr(metadata, 'license', 'N/A'))
print('Keywords:', getattr(metadata, 'keywords', 'N/A'))
print('Authors:', getattr(metadata, 'author', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs. Each record set, field, and column is uniquely referenced by its `@id`.

We'll list the available record sets and their fields. These IDs are used for data extraction and analysis.

In [ ]:
# List all record sets in the dataset and their fields
record_sets = dataset.record_sets()

record_set_ids = []
for rs in record_sets:
    print(f'RecordSet @id: {rs["@id"]}')
    record_set_ids.append(rs["@id"])
    print(f'  Name: {rs.get("name", "N/A")}')
    if "fields" in rs:
        print('  Fields:')
        for field in rs["fields"]:
            print(f'    Field @id: {field["@id"]}, Name: {field.get("name", field["@id"])}, DataType: {field.get("dataType", "N/A")}')
    print('---')

# For demonstration, print out the IDs for subsequent steps
print('Record Set IDs:', record_set_ids)

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

For demonstration, we'll create dataframes for each available record set.

In [ ]:
# Extract records for each record set and store in DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f'RecordSet {rs_id}, Columns: {df.columns.tolist()}')
        print(df.head(2))
    else:
        print(f'RecordSet {rs_id} has no records loaded.')

# Select the first record set with data for demonstration
if dataframes:
record_set_demo_id = list(dataframes.keys())[0]
demo_df = dataframes[record_set_demo_id]
print('Using RecordSet:', record_set_demo_id)
else:
print('No dataframes were loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps. All fields are referenced by their `@id`. We will demonstrate outlier filtering, normalization, and grouping for numeric and categorical fields.

Below, replace `<numeric_field_id>` and `<group_field_id>` with actual IDs from the previous overview.

In [ ]:
# --- EDA ---
# Identify a numeric field and group field by their @id (replace as appropriate)
numeric_field_id = None
group_field_id = None
# Try to infer numeric/group field candidates
for col in demo_df.columns:
    # Typical heuristics: look for columns named with 'age', 'interval', 'count', else pick first int/float
    if 'age' in col.lower() or 'interval' in col.lower():
        numeric_field_id = col
    if 'sex' in col.lower() or 'anatomical' in col.lower() or 'msi' in col.lower():
        group_field_id = col
if not numeric_field_id:
    # Fallback: try to choose a column with numerical dtype
    for col in demo_df.columns:
        if pd.api.types.is_numeric_dtype(demo_df[col]):
            numeric_field_id = col
            break
if not group_field_id:
    # Fallback: try to choose a column with object (category/string) dtype
    for col in demo_df.columns:
        if pd.api.types.is_object_dtype(demo_df[col]):
            group_field_id = col
            break
print(f'Numeric field selected @id: {numeric_field_id}')
print(f'Group field selected @id: {group_field_id}')

# Filter records (example: values > threshold)
if numeric_field_id is not None:
    threshold = demo_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(demo_df[numeric_field_id]) else 10
    filtered_df = demo_df[demo_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric field found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields using their `@id`. Common plots include a histogram and boxplot for numeric fields, and a barplot for categorical analysis.

In [ ]:
# Visualization
if numeric_field_id is not None and group_field_id is not None:
    plt.figure(figsize=(10,4))
    sns.histplot(demo_df[numeric_field_id], kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=demo_df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

    # Barplot of group sizes
    plt.figure(figsize=(8,3))
    demo_df[group_field_id].value_counts().plot(kind='bar')
    plt.title(f'Record count per {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print('Insufficient fields for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and visualize the FAIR^2 dataset using the `mlcroissant` library by referencing entities entirely by their `@id`. We extracted metadata, identified record sets and fields, performed exploratory data analysis, and visualized key distributions. For more advanced analyses, further exploration of domain-specific fields using their `@id` is encouraged.

#### For more information about Croissant schema and FAIR datasets, see:
- [mlcroissant documentation](https://mlcommons.github.io/croissant/)
- [FAIR^2 dataset registry](https://sen.science/doi/10.71728/senscience.qs2f-h81p)